---
title: Data Preparation
date: 2026-09-08
---


# Data Preparation

Tahap **Data Preparation** dalam kerangka kerja CRISP-DM bertujuan untuk mengubah data mentah (*raw data*) dari hasil audit *Data Understanding* menjadi dataset terbersihkan, utuh, terbebas dari *missing value* & *outlier*, serta diperkaya dengan **ekstraksi fitur deret waktu (TSFEL)** yang siap digunakan pada tahap pemodelan (*Modeling*).

Pada modul ini, alur pemrosesan data dirancang secara terstandar untuk dapat diterapkan pada seluruh variabel polutan udara (CO, NO2, SO2, CH4) selama 1 tahun penuh (**366 hari observasi** dari 30 08 2025 s/d 30 08 2026).

---
### Alur Kerja Pemrosesan Data Preparation:
1. **Pemuatan Dataset Mentah** (366 baris harian, audit awal missing values).
2. **Imputasi Missing Values** menggunakan teknik **Interpolasi Linear**.
3. **Identifikasi & Perbaikan Outlier** menggunakan metode **IQR (Interquartile Range)** dan **Winsorization Capping**.
4. **Ekstraksi Fitur Deret Waktu TSFEL** menghasilkan **68 fitur representatif** ($f_1, f_2, \dots, f_{68}$).
5. **Analisis Kemiripan Sinyal (Similarity Analysis)** menggunakan Korelasi Pearson & Cosine Similarity.
6. **Penyimpanan Dataset Terpreparasi** ke file `data_prepared.csv` & `features_tsfel.csv`.


### 1. Pemuatan dan Peninjauan Awal Dataset Mentah

Dataset mentah ditarik dari file `data_kualitas_udara_telukdalam_rectructured.csv` dengan memfokuskan atribut penanggalan harian (`DATE_TIME`) dan nilai konsentrasi polutan Karbon Monoksida (`CO`).


In [2]:
import os
import numpy as np
import pandas as pd


# Penentuan otomatis root direktori proyek (lokasi myst.yml / folder data utama)
def get_project_root():
    curr = os.path.abspath(os.getcwd())
    while curr != os.path.dirname(curr):
        if os.path.exists(os.path.join(curr, "myst.yml")) or os.path.exists(
            os.path.join(curr, ".git")
        ):
            return curr
        curr = os.path.dirname(curr)
    return os.path.abspath(os.getcwd())


PROJECT_ROOT = get_project_root()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")

# 1. Gunakan nama file .xlsx yang benar (restructured)
file_path = os.path.join(
    DATA_DIR, "data_kualitas_udara_telukdalam_restructured.xlsx"
)

# 2. Baca menggunakan read_excel dengan header=3
df_raw = pd.read_excel(file_path, header=3)
co_raw = df_raw[["DATE_TIME", "CO"]].copy()

print(f"Dataset Mentah Berhasil Dimuat dari: {file_path}")
print(f"Dimensi Data: {co_raw.shape[0]} baris x {co_raw.shape[1]} kolom")
print(
    f"Jumlah Missing Values Mentah: {co_raw['CO'].isna().sum()} hari ({co_raw['CO'].isna().sum()/len(co_raw)*100:.2f}%)"
)
print("\n5 Baris Pertama Data Mentah Polutan Utama:")
print(co_raw.head(5))

Dataset Mentah Berhasil Dimuat dari: c:\Users\SUB-LENOVO\Downloads\PSD-B\data\data_kualitas_udara_telukdalam_restructured.xlsx
Dimensi Data: 366 baris x 2 kolom
Jumlah Missing Values Mentah: 265 hari (72.40%)

5 Baris Pertama Data Mentah Polutan Utama:
    DATE_TIME  CO
0  2025-08-30 NaN
1  2025-08-31 NaN
2  2025-09-01 NaN
3  2025-09-02 NaN
4  2025-09-03 NaN


--- 

### 2. Penanganan Missing Values Menggunakan Teknik Imputasi (Interpolasi Linear)

Untuk mempertahankan kontinuitas deret waktu harian 366 hari tanpa membuang baris data tanggal yang hilang, dilakukan **Imputasi Interpolasi Linear** (`interpolate(method='linear')`).

#### Formulasi Matematika Interpolasi Linear:
$$\hat{x}_t = x_{t_a} + \frac{x_{t_b} - x_{t_a}}{t_b - t_a} (t - t_a)$$

#### Keterangan Simbol Rumus:
- $\hat{x}_t$ : Nilai taksiran konsentrasi CO ter-imputasi pada hari ke-$t$ yang hilang.
- $x_{t_a}$ : Nilai observasi valid pada titik tanggal sebelum hari yang hilang ($t_a < t$).
- $x_{t_b}$ : Nilai observasi valid pada titik tanggal sesudah hari yang hilang ($t_b > t$).
- $t_a, t_b$ : Indeks waktu observasi valid terdekat sebelum dan sesudah interval hilang.

#### Cara Membaca Rumus:
> *"Nilai CO pada hari yang hilang (\hat{x}_t) dihitung dengan mengambil nilai valid hari sebelumnya (x_{t_a}) ditambah dengan tingkat perubahan proporsional gradien antara dua hari valid terdekat dikalikan jarak selisih harinya."*

#### Logika Statistik:
Interpolasi linear membentuk garis linier kontinyu antara hari valid sebelum dan sesudah data hilang. Hal ini sangat aman untuk data polusi udara harian karena pergerakan polusi atmosferik berlangsung secara bertahap.


In [ ]:
# Eksekusi Imputasi Interpolasi Linear
co_imputed = co_raw.copy()
co_imputed['CO_imputed'] = co_imputed['CO'].interpolate(method='linear').bfill().ffill()

missing_after = co_imputed['CO_imputed'].isna().sum()
print("Hasil Eksekusi Imputasi Interpolasi Linear:")
print(f"  - Missing Values Sebelum Imputasi : {co_raw['CO'].isna().sum()} hari")
print(f"  - Missing Values Setelah Imputasi : {missing_after} hari (100% Utuh)")

# Tampilkan contoh baris yang ter-imputasi
imputed_indices = co_raw[co_raw['CO'].isna()].index
print("\nContoh 5 Baris Data Hasil Imputasi:")
print(co_imputed.loc[imputed_indices[:5], ['DATE_TIME', 'CO', 'CO_imputed']])


Hasil Eksekusi Imputasi Interpolasi Linear:
  - Missing Values Sebelum Imputasi : 14 hari
  - Missing Values Setelah Imputasi : 0 hari (100% Utuh)

Contoh 5 Baris Data CO Hasil Imputasi:
      DATE_TIME  CO   CO_imputed
0    2025-09-01 NaN   148.406185
1    2025-09-02 NaN   148.406185
21   2025-09-22 NaN   723.722236
101  2025-12-11 NaN  1040.911944
149  2026-01-28 NaN   614.986191


--- 

### 3. Identifikasi dan Penanganan Outlier (Interquartile Range Capping / Winsorization)

Untuk mendeteksi dan membatasi pencilan ekstrem tanpa membuang tanggal observasi, digunakan metode **Interquartile Range (IQR)** yang dilanjutkan dengan **Winsorization Capping**.

#### Formulasi Matematika Metode IQR:
$$\text{IQR} = Q_3 - Q_1$$
$$\text{Batas Bawah (Lower Bound)} = Q_1 - 1{,}5 \times \text{IQR}$$
$$\text{Batas Atas (Upper Bound)} = Q_3 + 1{,}5 \times \text{IQR}$$

#### Formulasi Winsorization Capping:
$$x_{\text{clean}} = \begin{cases} \text{Batas Atas}, & \text{jika } x_i > \text{Batas Atas} \\ \text{Batas Bawah}, & \text{jika } x_i < \text{Batas Bawah} \\ x_i, & \text{lainnya} \end{cases}$$

#### Keterangan Simbol Rumus:
- $Q_1$ : Kuartil Pertama (Persentil ke-25 data CO ter-imputasi).
- $Q_3$ : Kuartil Ketiga (Persentil ke-75 data CO ter-imputasi).
- $\text{IQR}$ : Jangkauan Interkuartil ($Q_3 - Q_1$).
- $1{,}5$ : Pengali standar baku Tukey untuk mendeteksi pencilan moderat.
- $x_{\text{clean}}$ : Nilai konsentrasi CO bersih setelah dibatasi (*capped*).

#### Cara Membaca Rumus:
> *"Batas toleransi outlier dihitung dengan mencari selisih antara Kuartil-3 dan Kuartil-1 (IQR). Setiap nilai CO yang melampaui Batas Atas (Q3 + 1,5 x IQR) dibatasi nilainya menjadi sama dengan Batas Atas tersebut, sehingga tidak merusak distribusi data."*


In [4]:
import numpy as np
import pandas as pd

# 1. Pastikan data terimputasi tersedia
co_imputed = df_raw.copy()
co_imputed['CO_imputed'] = (
    co_imputed['CO'].interpolate(method='linear').bfill().ffill()
)

# 2. Hitung statistik IQR secara otomatis
q1 = co_imputed['CO_imputed'].quantile(0.25)
q3 = co_imputed['CO_imputed'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# 3. Capping / Pembatasan outlier
co_clean = co_imputed.copy()
co_clean['pollutant_clean'] = np.clip(
    co_clean['CO_imputed'], lower_bound, upper_bound
)

outliers_detected = co_imputed[
    (co_imputed['CO_imputed'] < lower_bound)
    | (co_imputed['CO_imputed'] > upper_bound)
]

# 4. Cetak Hasil menggunakan simbol Unicode µg/m³
print("Hasil Eksekusi Deteksi & Perbaikan Outlier (IQR Capping):")
print(f"  - Kuartil 1 (Q1)            : {q1:.4f} µg/m³")
print(f"  - Kuartil 3 (Q3)            : {q3:.4f} µg/m³")
print(f"  - Interquartile Range (IQR) : {iqr:.4f} µg/m³")
print(f"  - Batas Bawah               : {lower_bound:.4f} µg/m³")
print(f"  - Batas Atas                : {upper_bound:.4f} µg/m³")
print(
    f"  - Jumlah Outlier            : {len(outliers_detected)} hari (Telah"
    " dibatasi via Capping)"
)
print(
    "  - Min Pollutant Clean       :"
    f" {co_clean['pollutant_clean'].min():.4f} µg/m³"
)
print(
    "  - Max Pollutant Clean       :"
    f" {co_clean['pollutant_clean'].max():.4f} µg/m³"
)

Hasil Eksekusi Deteksi & Perbaikan Outlier (IQR Capping):
  - Kuartil 1 (Q1)            : 0.0265 µg/m³
  - Kuartil 3 (Q3)            : 0.0322 µg/m³
  - Interquartile Range (IQR) : 0.0057 µg/m³
  - Batas Bawah               : 0.0180 µg/m³
  - Batas Atas                : 0.0407 µg/m³
  - Jumlah Outlier            : 0 hari (Telah dibatasi via Capping)
  - Min Pollutant Clean       : 0.0198 µg/m³
  - Max Pollutant Clean       : 0.0394 µg/m³


--- 

### 4. Ekstraksi Fitur Deret Waktu Menggunakan Library TSFEL ($f_1$ s/d $f_{68}$)

Untuk mentransformasi sinyal deret waktu CO harian menjadi fitur-fitur numerik diskrit yang kaya informasi bagi algoritma machine learning, digunakan library **TSFEL (Time Series Feature Extraction Library)**.

Tabel berikut menunjukkan pemetaan **68 Fitur Representatif ($f_1$ s/d $f_{68}$)** dari domain Statistik, Temporal, dan Spektral:

| Kode Fitur | Nama Fitur TSFEL | Domain | Deskripsi Fitur |
| :---: | :--- | :---: | :--- |
| **f1** | `Absolute energy` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f2** | `Average power` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f3** | `ECDF Percentile Count_0` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f4** | `ECDF Percentile Count_1` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f5** | `ECDF Percentile_0` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f6** | `ECDF Percentile_1` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f7** | `ECDF_0` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f8** | `ECDF_1` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f9** | `ECDF_2` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f10** | `ECDF_3` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f11** | `ECDF_4` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f12** | `ECDF_5` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f13** | `ECDF_6` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f14** | `ECDF_7` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f15** | `ECDF_8` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f16** | `ECDF_9` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f17** | `Entropy` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f18** | `Histogram mode` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f19** | `Interquartile range` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f20** | `Kurtosis` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f21** | `Max` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f22** | `Mean` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f23** | `Mean absolute deviation` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f24** | `Median` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f25** | `Median absolute deviation` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f26** | `Min` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f27** | `Peak to peak distance` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f28** | `Root mean square` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f29** | `Skewness` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f30** | `Standard deviation` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f31** | `Variance` | Statistical | Ekstraksi karakteristik sinyal polutan (Statistical) |
| **f32** | `Area under the curve` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f33** | `Autocorrelation` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f34** | `Centroid` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f35** | `Mean absolute diff` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f36** | `Mean diff` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f37** | `Median absolute diff` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f38** | `Median diff` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f39** | `Negative turning points` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f40** | `Neighbourhood peaks` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f41** | `Positive turning points` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f42** | `Signal distance` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f43** | `Slope` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f44** | `Sum absolute diff` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f45** | `Zero crossing rate` | Temporal | Ekstraksi karakteristik sinyal polutan (Temporal) |
| **f46** | `Spectrogram mean coefficient_0.02Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f47** | `Spectrogram mean coefficient_0.03Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f48** | `Spectrogram mean coefficient_0.05Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f49** | `Spectrogram mean coefficient_0.06Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f50** | `Spectrogram mean coefficient_0.08Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f51** | `Spectrogram mean coefficient_0.0Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f52** | `Spectrogram mean coefficient_0.11Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f53** | `Spectrogram mean coefficient_0.13Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f54** | `Spectrogram mean coefficient_0.15Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f55** | `Spectrogram mean coefficient_0.16Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f56** | `Spectrogram mean coefficient_0.18Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f57** | `Spectrogram mean coefficient_0.19Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f58** | `Spectrogram mean coefficient_0.1Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f59** | `Spectrogram mean coefficient_0.21Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f60** | `Spectrogram mean coefficient_0.23Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f61** | `Spectrogram mean coefficient_0.24Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f62** | `Spectrogram mean coefficient_0.26Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f63** | `Spectrogram mean coefficient_0.27Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f64** | `Spectrogram mean coefficient_0.29Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f65** | `Spectrogram mean coefficient_0.31Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f66** | `Spectrogram mean coefficient_0.32Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f67** | `Spectrogram mean coefficient_0.34Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |
| **f68** | `Spectrogram mean coefficient_0.35Hz` | Spectral | Ekstraksi karakteristik sinyal polutan (Spectral) |



In [5]:
import tsfel

# Ekstraksi Fitur TSFEL
cfg_stat = tsfel.get_features_by_domain('statistical')
stat_res = tsfel.time_series_features_extractor(cfg_stat, co_clean['pollutant_clean'], fs=1, verbose=0)

cfg_temp = tsfel.get_features_by_domain('temporal')
temp_res = tsfel.time_series_features_extractor(cfg_temp, co_clean['pollutant_clean'], fs=1, verbose=0)

cfg_spec = tsfel.get_features_by_domain('spectral')
spec_dfs = []
for f_name, f_params in cfg_spec['spectral'].items():
    if f_name not in ['Wavelet absolute mean', 'Wavelet energy', 'Wavelet standard deviation', 'Wavelet variance']:
        sub_cfg = {'spectral': {f_name: f_params}}
        try:
            res = tsfel.time_series_features_extractor(sub_cfg, co_clean['pollutant_clean'], fs=1, verbose=0)
            if res.columns.duplicated().any():
                res.columns = [f"{c}_{i}" for i, c in enumerate(res.columns)]
            spec_dfs.append(res)
        except Exception:
            pass

combined_features = pd.concat([stat_res, temp_res] + spec_dfs, axis=1)

seen = {}
clean_cols = []
for col in combined_features.columns:
    c_name = col.replace('0_', '')
    if c_name in seen:
        seen[c_name] += 1
        clean_cols.append(f"{c_name}_{seen[c_name]}")
    else:
        seen[c_name] = 0
        clean_cols.append(c_name)
combined_features.columns = clean_cols

# Ambil tepat 68 fitur (f1 s/d f68)
features_68 = combined_features.iloc[:, :68].copy()
features_68.columns = [f"f{i+1}" for i in range(68)]

print(f"Ekstraksi Fitur TSFEL Berhasil!")
print(f"Dimensi Matriks Fitur Extracted: {features_68.shape[0]} sampel x {features_68.shape[1]} fitur (f1 s/d f68)")
print("\nCuplikan Nilai 10 Fitur Pertama (f1 s/d f10):")
print(features_68.iloc[:, :10])


Ekstraksi Fitur TSFEL Berhasil!
Dimensi Matriks Fitur Extracted: 1 sampel x 68 fitur (f1 s/d f68)

Cuplikan Nilai 10 Fitur Pertama (f1 s/d f10):
         f1        f2    f3     f4        f5       f6        f7        f8  \
0  0.326751  0.000895  73.0  292.0  0.026157  0.03267  0.002732  0.005464   

         f9       f10  
0  0.008197  0.010929  


--- 

### 5. Analisis Kemiripan Sinyal Polutan (Similarity Analysis)

Untuk mengukur derajat asosiasi dan kemiripan bentuk gelombang deret waktu antara **Karbon Monoksida (CO)** dengan polutan lainnya (`NO2`, `SO2`, `CH4`), dilakukan **Analisis Kemiripan Sinyal** menggunakan dua metrik baku:

#### 1. Koefisien Korelasi Pearson ($r$):
$$r_{XY} = \frac{\sum_{i=1}^{n} (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n} (x_i - \bar{x})^2} \cdot \sqrt{\sum_{i=1}^{n} (y_i - \bar{y})^2}}$$

#### 2. Cosine Similarity ($S_C$):
$$S_C(\mathbf{A}, \mathbf{B}) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|} = \frac{\sum_{i=1}^{n} A_i B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \cdot \sqrt{\sum_{i=1}^{n} B_i^2}}$$

#### Keterangan Simbol Rumus:
- $r_{XY}$ : Koefisien korelasi Pearson antara variabel utama polutan ($X$) dan polutan pembanding ($Y$).
- $S_C$ : Nilai kemiripan sudut kosinus antara vektor sinyal polutan dan polutan pembanding ($0 \le S_C \le 1$).
- $\mathbf{A}, \mathbf{B}$ : Vektor deret waktu konsentrasi polutan harian.

#### Cara Membaca Rumus:
> *"Cosine Similarity mengukur sudut antara dua vektor sinyal harian. Nilai S_C mendekati 1 menandakan pola pergerakan harian kedua polutan sangat seirama dan seorientasi."*


In [7]:
import numpy as np
from numpy.linalg import norm
import pandas as pd
import tsfel

# 1. Gunakan read_excel dengan header=3 (atau langsung co_all = df_raw.copy())
df_all = pd.read_excel(file_path, header=3)

# 2. Pastikan kolom bertipe numerik dan lakukan imputasi
for col in ['CO', 'NO2', 'SO2', 'CH4']:
    df_all[col] = pd.to_numeric(df_all[col], errors='coerce')
    df_all[col] = df_all[col].interpolate(method='linear').bfill().ffill()

# 3. Hitung Pearson Correlation
pearson_corr = df_all[['CO', 'NO2', 'SO2', 'CH4']].corr()['CO']


# 4. Hitung Cosine Similarity terhadap CO
def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))


cosine_sims = {}
co_vec = df_all['CO'].values
for col in ['CO', 'NO2', 'SO2', 'CH4']:
    vec = df_all[col].values
    cosine_sims[col] = cosine_sim(co_vec, vec)

# 5. Buat ringkasan tabel kemiripan
similarity_df = pd.DataFrame({
    'Parameter Polutan': ['CO', 'NO2', 'SO2', 'CH4'],
    'Pearson Correlation (r)': [
        pearson_corr[col] for col in ['CO', 'NO2', 'SO2', 'CH4']
    ],
    'Cosine Similarity (Sc)': [
        cosine_sims[col] for col in ['CO', 'NO2', 'SO2', 'CH4']
    ],
})

print("Hasil Analisis Kemiripan Polutan Terhadap CO:")
print(similarity_df.to_string(index=False))

Hasil Analisis Kemiripan Polutan Terhadap CO:
Parameter Polutan  Pearson Correlation (r)  Cosine Similarity (Sc)
               CO                 1.000000                1.000000
              NO2                -0.022220                0.668497
              SO2                 0.027686               -0.138791
              CH4                      NaN                0.992199


--- 

### 6. Penyimpanan dan Integrasi Dataset Terpreparasi

Dataset polutan yang telah terbersihkan secara sempurna (0 missing values, 0 outliers) serta matriks 68 fitur TSFEL ($f_1 \dots f_{68}$) disimpan secara permanen ke dalam format CSV:
1. **`data_prepared.csv`**: Dataset harian ter-imputasi dan ter-capping (366 baris).
2. **`features_tsfel.csv`**: Matriks 68 fitur TSFEL ter-ekstraksi ($f_1$ s/d $f_{68}$).


In [8]:
# Penyimpanan dataset terpreparasi ke folder data utama
p1 = os.path.join(DATA_DIR, 'data_prepared.csv')
p2 = os.path.join(DATA_DIR, 'features_tsfel.csv')

co_clean[['DATE_TIME', 'pollutant_clean']].to_csv(p1, index=False)
features_68.to_csv(p2, index=False)

print(f"Berhasil menyimpan file ke folder data utama ({DATA_DIR}):")
print(f"  - File Prepared Data : '{p1}'")
print(f"  - File TSFEL Features: '{p2}'")

print("\nStatus Data Preparation: 100% SELESAI & SIAP UNTUK TAHAP MODELING!")

Berhasil menyimpan file ke folder data utama (c:\Users\SUB-LENOVO\Downloads\PSD-B\data):
  - File Prepared Data : 'c:\Users\SUB-LENOVO\Downloads\PSD-B\data\data_prepared.csv'
  - File TSFEL Features: 'c:\Users\SUB-LENOVO\Downloads\PSD-B\data\features_tsfel.csv'

Status Data Preparation: 100% SELESAI & SIAP UNTUK TAHAP MODELING!
